In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 jinja2==3.1.4 rapidfuzz==3.9.7 tqdm==4.66.5


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import unicodedata

import pandas as pd
from jinja2 import Environment, select_autoescape
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm


In [ ]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
OUTPUT_DIR = BASE_DIR / "output"
CACHE_DIR = BASE_DIR / "cache"
RAW_DIR = BASE_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"

for directory in [BASE_DIR, OUTPUT_DIR, CACHE_DIR, RAW_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")


In [ ]:
INPUT_CONTEXT_COLUMNS = ["estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input"]
EVIDENCE_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
    "source", "source_type", "source_country", "source_official", "source_category", "source_reliability",
    "matched_record_name", "matched_record_normalized_name", "matched_alias", "matched_entity_type", "matched_role", "matched_position", "matched_agency", "matched_state", "matched_municipality", "matched_country", "matched_identifier", "matched_company", "matched_rfc", "matched_curp",
    "match_score_pct", "match_strength", "match_method", "matched_fields", "conflicting_fields", "requires_review", "identity_confidence_pct", "signal_type", "signal_category", "severity", "risk_layer", "risk_level_candidate", "confidence_pct",
    "evidence_title", "evidence_summary", "evidence_snippet", "evidence_keywords", "evidence_date", "evidence_url", "source_url", "search_query", "search_engine", "retrieved_at", "raw_file_path", "raw_file_sha256",
    "involvement_summary", "explanation", "limitations", "recommended_action",
]
MEDIA_EXTRA_COLUMNS = ["media_domain", "media_language", "media_country", "keyword_hits", "adverse_keyword_count", "official_domain_flag", "primary_source_flag"]
MASTER_EVIDENCE_COLUMNS = EVIDENCE_COLUMNS + MEDIA_EXTRA_COLUMNS
MASTER_SUMMARY_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
    "total_signals", "official_signals", "sanctions_signals", "mexico_public_signals", "media_signals", "search_signals", "sat_69b_candidates", "sancionados_candidates", "compranet_candidates", "declaranet_candidates", "ofac_candidates", "un_candidates", "duckduckgo_results", "gdelt_results",
    "max_match_score_pct", "avg_match_score_pct", "max_identity_confidence_pct", "avg_identity_confidence_pct", "max_severity", "risk_level_candidate", "requires_review", "recommended_action",
    "top_1_source", "top_1_signal_type", "top_1_match_score_pct", "top_1_evidence_url", "top_1_involvement_summary",
    "top_2_source", "top_2_signal_type", "top_2_match_score_pct", "top_2_evidence_url", "top_2_involvement_summary",
    "top_3_source", "top_3_signal_type", "top_3_match_score_pct", "top_3_evidence_url", "top_3_involvement_summary",
    "final_explanation", "limitations", "generated_at",
]
URL_CATALOG_COLUMNS = ["evidence_id", "source", "source_type", "source_official", "evidence_url", "source_url", "evidence_title", "evidence_date", "retrieved_at", "raw_file_path", "raw_file_sha256", "used_for_seed_ids", "used_for_names", "notes"]
COVERAGE_COLUMNS = ["source", "source_type", "attempted", "success", "rows_downloaded_or_loaded", "rows_parsed", "matches_found", "unique_people_with_hits", "runtime_seconds", "error_message", "notes"]
DISCLAIMER = "Este reporte contiene senales publicas y candidatos de coincidencia. No constituye determinacion legal, penal ni regulatoria. Cuando el insumo contiene solo nombre, los resultados deben tratarse como candidatos sujetos a revision humana."
PARTICLES = {"de", "del", "la", "las", "los", "el", "y"}
RISK_RANK = {"critical_candidate": 0, "high_review": 1, "medium_review": 2, "low_signal": 3, "insufficient_evidence": 4}
SEVERITY_RANK = {"official_sensitive_candidate": 0, "official_strong": 1, "official_mexico_public_record_candidate": 2, "media_candidate": 3, "search_only": 4, "": 9}

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = "".join(ch for ch in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(ch)).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_name(value):
    return " ".join(normalize_text(value).split())

def stable_hash(value):
    return hashlib.sha256(normalize_name(value).encode("utf-8")).hexdigest()[:16]

def empty_master_evidence_df():
    return pd.DataFrame(columns=MASTER_EVIDENCE_COLUMNS)

def input_has_only_name(seed):
    return bool(seed.get("has_only_name", False)) or not any(str(seed.get(col, "")).strip() for col in INPUT_CONTEXT_COLUMNS)

def bool_series(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes", "si"])


In [ ]:
def fallback_seed_dataframe():
    print("ADVERTENCIA: no existe output/00_peps_normalized.csv; se crean datos minimos de prueba para esta corrida.")
    names = ["Claudia Sheinbaum Pardo", "Andres Manuel Lopez Obrador", "Marcelo Ebrard Casaubon", "Adan Augusto Lopez Hernandez", "Ricardo Monreal Avila", "Xochitl Galvez Ruiz", "Samuel Garcia Sepulveda", "Clara Brugada Molina", "Omar Garcia Harfuch", "Luisa Maria Alcalde Lujan"]
    return pd.DataFrame([{"seed_id": stable_hash(name), "nombre_persona_input": name, "normalized_name_input": normalize_name(name), "estado_input": "", "municipio_input": "", "puesto_input": "", "dependencia_input": "", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "has_only_name": True} for name in names])

input_file = OUTPUT_DIR / "00_peps_normalized.csv"
seeds_df = pd.read_csv(input_file, encoding="utf-8-sig").fillna("") if input_file.exists() else fallback_seed_dataframe()
for col in ["seed_id", "nombre_persona_input", "normalized_name_input"] + INPUT_CONTEXT_COLUMNS + ["has_only_name"]:
    if col not in seeds_df.columns:
        seeds_df[col] = ""
print(f"Personas base: {len(seeds_df):,}")
display(seeds_df.head(20))


In [ ]:
EVIDENCE_FILES = [
    OUTPUT_DIR / "01_sanciones_ofac_onu_evidence.csv",
    OUTPUT_DIR / "02_fuentes_mexico_evidence.csv",
    OUTPUT_DIR / "03_adverse_media_evidence.csv",
]
frames = []
for path in EVIDENCE_FILES:
    if path.exists():
        try:
            df = pd.read_csv(path, encoding="utf-8-sig").fillna("")
            for column in MASTER_EVIDENCE_COLUMNS:
                if column not in df.columns:
                    df[column] = ""
            frames.append(df[MASTER_EVIDENCE_COLUMNS])
            print(f"Leido: {path.name} ({len(df):,} filas)")
        except Exception as exc:
            print(f"No se pudo leer {path.name}: {exc}")
    else:
        print(f"Falta opcional: {path.name}; se continua sin fallar.")
master_evidence_df = pd.concat(frames, ignore_index=True) if frames else empty_master_evidence_df()
if not master_evidence_df.empty:
    for col in ["match_score_pct", "identity_confidence_pct", "confidence_pct"]:
        master_evidence_df[col] = pd.to_numeric(master_evidence_df[col], errors="coerce").fillna(0.0)
    master_evidence_df = master_evidence_df.drop_duplicates(subset=["seed_id", "source", "signal_type", "evidence_url", "matched_record_name", "search_query"], keep="first")
master_evidence_path = OUTPUT_DIR / "04_master_evidence.csv"
master_evidence_df.to_csv(master_evidence_path, index=False, encoding="utf-8-sig")
print(f"Master evidence guardado: {master_evidence_path}")
display(master_evidence_df.head(30))


In [ ]:
def top_evidence_rows(subset, n=3):
    if subset.empty:
        return []
    temp = subset.copy()
    temp["_severity_rank"] = temp["severity"].astype(str).map(lambda x: SEVERITY_RANK.get(x, 9))
    temp["_match"] = pd.to_numeric(temp["match_score_pct"], errors="coerce").fillna(0)
    temp["_confidence"] = pd.to_numeric(temp["identity_confidence_pct"], errors="coerce").fillna(0)
    return temp.sort_values(["_severity_rank", "_match", "_confidence"], ascending=[True, False, False]).head(n).to_dict(orient="records")

def risk_for_person(seed, subset):
    if subset.empty:
        return "insufficient_evidence"
    max_score = pd.to_numeric(subset["match_score_pct"], errors="coerce").fillna(0).max()
    max_conf = pd.to_numeric(subset["identity_confidence_pct"], errors="coerce").fillna(0).max()
    stype = subset["source_type"].astype(str).str.lower()
    category = subset["source_category"].astype(str).str.lower()
    signal = subset["signal_type"].astype(str).str.lower()
    has_sanctions = stype.str.contains("sanctions").any() or signal.str.contains("ofac|un_sanctions").any()
    official_count = int(bool_series(subset["source_official"]).sum())
    media_count = int(category.str.contains("adverse_media").sum())
    if has_sanctions and max_score >= 95 and max_conf >= 85 and not input_has_only_name(seed):
        return "critical_candidate"
    if has_sanctions and max_score >= 88:
        return "high_review"
    if official_count >= 2 and max_score >= 88:
        return "high_review"
    if official_count >= 1 or media_count >= 3:
        return "medium_review"
    return "low_signal"

def action_for_risk(risk, subset):
    if risk in {"critical_candidate", "high_review"}:
        return "enhanced_due_diligence_review"
    if risk == "medium_review":
        return "manual_review"
    if risk == "low_signal":
        if not subset.empty and subset["source_category"].astype(str).str.contains("adverse_media", case=False, na=False).any():
            return "verify_primary_source"
        return "manual_review"
    return "no_action"

summary_rows = []
generated_at = now_utc()
for _, seed in seeds_df.iterrows():
    subset = master_evidence_df[master_evidence_df["seed_id"].astype(str) == str(seed.get("seed_id", ""))] if not master_evidence_df.empty else empty_master_evidence_df()
    total = len(subset)
    source = subset["source"].astype(str) if total else pd.Series(dtype=str)
    signal = subset["signal_type"].astype(str) if total else pd.Series(dtype=str)
    category = subset["source_category"].astype(str) if total else pd.Series(dtype=str)
    stype = subset["source_type"].astype(str) if total else pd.Series(dtype=str)
    scores = pd.to_numeric(subset["match_score_pct"], errors="coerce").fillna(0) if total else pd.Series(dtype=float)
    confs = pd.to_numeric(subset["identity_confidence_pct"], errors="coerce").fillna(0) if total else pd.Series(dtype=float)
    top_rows = top_evidence_rows(subset, 3)
    risk = risk_for_person(seed, subset)
    row = {
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""),
        "normalized_name_input": seed.get("normalized_name_input", normalize_name(seed.get("nombre_persona_input", ""))),
        "estado_input": seed.get("estado_input", ""), "municipio_input": seed.get("municipio_input", ""), "puesto_input": seed.get("puesto_input", ""), "dependencia_input": seed.get("dependencia_input", ""),
        "periodo_inicio_input": seed.get("periodo_inicio_input", ""), "periodo_fin_input": seed.get("periodo_fin_input", ""), "partido_input": seed.get("partido_input", ""), "rfc_input": seed.get("rfc_input", ""), "curp_input": seed.get("curp_input", ""), "empresa_relacionada_input": seed.get("empresa_relacionada_input", ""), "alias_input": seed.get("alias_input", ""),
        "total_signals": total,
        "official_signals": int(bool_series(subset["source_official"]).sum()) if total else 0,
        "sanctions_signals": int(stype.str.contains("sanctions", case=False, na=False).sum() + signal.str.contains("ofac|un_sanctions", case=False, na=False).sum()) if total else 0,
        "mexico_public_signals": int(category.str.contains("mexico_public", case=False, na=False).sum()) if total else 0,
        "media_signals": int(category.str.contains("adverse_media", case=False, na=False).sum() + stype.str.contains("gdelt_media", case=False, na=False).sum()) if total else 0,
        "search_signals": int(stype.str.contains("search_result", case=False, na=False).sum()) if total else 0,
        "sat_69b_candidates": int(signal.str.contains("sat_69b", case=False, na=False).sum()) if total else 0,
        "sancionados_candidates": int(signal.str.contains("servidor_publico_sancionado", case=False, na=False).sum()) if total else 0,
        "compranet_candidates": int(signal.str.contains("contrato|compranet|proveedor|adjudicacion", case=False, na=False).sum()) if total else 0,
        "declaranet_candidates": int(signal.str.contains("declaranet|public_official", case=False, na=False).sum()) if total else 0,
        "ofac_candidates": int(source.str.contains("OFAC", case=False, na=False).sum()) if total else 0,
        "un_candidates": int(source.str.contains("UN", case=False, na=False).sum()) if total else 0,
        "duckduckgo_results": int(source.str.contains("DuckDuckGo", case=False, na=False).sum()) if total else 0,
        "gdelt_results": int(source.str.contains("GDELT", case=False, na=False).sum()) if total else 0,
        "max_match_score_pct": round(float(scores.max()), 2) if total else 0,
        "avg_match_score_pct": round(float(scores.mean()), 2) if total else 0,
        "max_identity_confidence_pct": round(float(confs.max()), 2) if total else 0,
        "avg_identity_confidence_pct": round(float(confs.mean()), 2) if total else 0,
        "max_severity": top_rows[0].get("severity", "") if top_rows else "",
        "risk_level_candidate": risk,
        "requires_review": bool(total > 0 or input_has_only_name(seed)),
        "recommended_action": action_for_risk(risk, subset),
        "final_explanation": "Sin hits en fuentes procesadas." if total == 0 else "Resumen conservador basado en evidencias tabulares; ninguna senal confirma identidad, responsabilidad legal ni conducta ilicita por si sola.",
        "limitations": "Input solo con nombre: limitar confianza y verificar homonimos." if input_has_only_name(seed) else "Verificar identidad con identificadores adicionales y fuente primaria antes de cualquier decision.",
        "generated_at": generated_at,
    }
    for idx in range(3):
        top = top_rows[idx] if idx < len(top_rows) else {}
        n = idx + 1
        row[f"top_{n}_source"] = top.get("source", "")
        row[f"top_{n}_signal_type"] = top.get("signal_type", "")
        row[f"top_{n}_match_score_pct"] = top.get("match_score_pct", "")
        row[f"top_{n}_evidence_url"] = top.get("evidence_url", "")
        row[f"top_{n}_involvement_summary"] = top.get("involvement_summary", "")
    summary_rows.append(row)
master_summary_df = pd.DataFrame(summary_rows, columns=MASTER_SUMMARY_COLUMNS)
master_summary_df["_risk_rank"] = master_summary_df["risk_level_candidate"].map(RISK_RANK).fillna(99)
master_summary_df = master_summary_df.sort_values(["_risk_rank", "total_signals", "max_match_score_pct"], ascending=[True, False, False]).drop(columns=["_risk_rank"])
master_summary_path = OUTPUT_DIR / "04_master_person_summary.csv"
master_summary_df.to_csv(master_summary_path, index=False, encoding="utf-8-sig")
print(f"Master person summary guardado: {master_summary_path}")
display(master_summary_df)


In [ ]:
catalog_rows = []
if not master_evidence_df.empty:
    group_cols = ["source", "source_type", "source_official", "evidence_url", "source_url", "evidence_title", "evidence_date", "retrieved_at", "raw_file_path", "raw_file_sha256"]
    temp = master_evidence_df.copy()
    temp["evidence_url"] = temp["evidence_url"].replace("", "no_url_available")
    for keys, group in temp.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        evidence_key = "|".join(str(row.get(col, "")) for col in ["source", "evidence_url", "evidence_title", "raw_file_sha256"])
        catalog_rows.append({
            "evidence_id": hashlib.sha256(evidence_key.encode("utf-8")).hexdigest()[:16],
            "source": row.get("source", ""), "source_type": row.get("source_type", ""), "source_official": row.get("source_official", ""),
            "evidence_url": "" if row.get("evidence_url") == "no_url_available" else row.get("evidence_url", ""),
            "source_url": row.get("source_url", ""), "evidence_title": row.get("evidence_title", ""), "evidence_date": row.get("evidence_date", ""),
            "retrieved_at": row.get("retrieved_at", ""), "raw_file_path": row.get("raw_file_path", ""), "raw_file_sha256": row.get("raw_file_sha256", ""),
            "used_for_seed_ids": "; ".join(sorted(set(group["seed_id"].astype(str)))),
            "used_for_names": "; ".join(sorted(set(group["nombre_persona_input"].astype(str)))),
            "notes": "URL/documento usado como evidencia candidata; revisar fuente primaria.",
        })
url_catalog_df = pd.DataFrame(catalog_rows, columns=URL_CATALOG_COLUMNS)
url_catalog_path = OUTPUT_DIR / "04_urls_evidence_catalog.csv"
url_catalog_df.to_csv(url_catalog_path, index=False, encoding="utf-8-sig")
print(f"Catalogo URLs guardado: {url_catalog_path}")
display(url_catalog_df.head(20))


In [ ]:
BENCHMARK_FILES = [OUTPUT_DIR / "00_benchmark_normalizacion.csv", OUTPUT_DIR / "01_benchmark_sanciones.csv", OUTPUT_DIR / "02_benchmark_fuentes_mexico.csv", OUTPUT_DIR / "03_benchmark_noticias.csv"]
benchmark_frames = []
for path in BENCHMARK_FILES:
    if path.exists():
        try:
            df = pd.read_csv(path, encoding="utf-8-sig").fillna("")
            df["benchmark_file"] = path.name
            benchmark_frames.append(df)
        except Exception as exc:
            print(f"No se pudo leer benchmark {path.name}: {exc}")
    else:
        print(f"Benchmark opcional faltante: {path.name}")
benchmark_general = pd.concat(benchmark_frames, ignore_index=True, sort=False) if benchmark_frames else pd.DataFrame()
for col in ["duration_seconds", "records_processed"]:
    if col not in benchmark_general.columns:
        benchmark_general[col] = 0
    benchmark_general[col] = pd.to_numeric(benchmark_general[col], errors="coerce").fillna(0)
coverage_rows = []
if not benchmark_general.empty and "source" in benchmark_general.columns:
    for source, group in benchmark_general.groupby("source", dropna=False):
        source_text = str(source)
        src_mask = master_evidence_df["source"].astype(str).eq(source_text) if not master_evidence_df.empty else pd.Series(dtype=bool)
        matches = int(src_mask.sum()) if not master_evidence_df.empty else 0
        people = int(master_evidence_df.loc[src_mask, "seed_id"].nunique()) if matches else 0
        parsed = group.loc[group["step"].astype(str).str.contains("parsing", case=False, na=False), "records_processed"].sum()
        loaded = group["records_processed"].sum()
        success = group["status"].astype(str).str.lower().isin(["ok", "cached"]).any()
        errors = "; ".join(group.loc[group["status"].astype(str).str.lower().eq("error"), "error_message"].astype(str).tolist())[:500]
        coverage_rows.append({
            "source": source_text,
            "source_type": master_evidence_df.loc[src_mask, "source_type"].iloc[0] if matches else "benchmark_only",
            "attempted": True,
            "success": bool(success or matches),
            "rows_downloaded_or_loaded": int(loaded),
            "rows_parsed": int(parsed),
            "matches_found": matches,
            "unique_people_with_hits": people,
            "runtime_seconds": round(float(group["duration_seconds"].sum()), 6),
            "error_message": errors,
            "notes": "; ".join(group.get("notes", pd.Series(dtype=str)).astype(str).head(5).tolist())[:500],
        })
for source in ["OFAC SDN", "OFAC Consolidated", "UN Consolidated", "SAT 69-B", "Servidores publicos sancionados", "CompraNet", "DeclaraNet", "DuckDuckGo", "GDELT"]:
    if source not in [row["source"] for row in coverage_rows]:
        src_mask = master_evidence_df["source"].astype(str).eq(source) if not master_evidence_df.empty else pd.Series(dtype=bool)
        coverage_rows.append({"source": source, "source_type": "not_run_or_no_benchmark", "attempted": False, "success": False, "rows_downloaded_or_loaded": 0, "rows_parsed": 0, "matches_found": int(src_mask.sum()) if not master_evidence_df.empty else 0, "unique_people_with_hits": int(master_evidence_df.loc[src_mask, "seed_id"].nunique()) if not master_evidence_df.empty and int(src_mask.sum()) else 0, "runtime_seconds": 0, "error_message": "", "notes": "No se encontro benchmark para esta fuente."})
coverage_df = pd.DataFrame(coverage_rows, columns=COVERAGE_COLUMNS)
coverage_path = OUTPUT_DIR / "04_source_coverage_summary.csv"
coverage_df.to_csv(coverage_path, index=False, encoding="utf-8-sig")
print(f"Coverage summary guardado: {coverage_path}")
display(coverage_df)


In [ ]:
total_time = float(benchmark_general["duration_seconds"].sum()) if not benchmark_general.empty else 0.0
slowest_notebook = benchmark_general.groupby("benchmark_file")["duration_seconds"].sum().sort_values(ascending=False).index[0] if not benchmark_general.empty and "benchmark_file" in benchmark_general.columns else ""
slowest_source = benchmark_general.groupby("source")["duration_seconds"].sum().sort_values(ascending=False).index[0] if not benchmark_general.empty and "source" in benchmark_general.columns else ""
records_processed = float(benchmark_general["records_processed"].sum()) if not benchmark_general.empty else 0.0
signals_generated = len(master_evidence_df)
signals_per_second = round(signals_generated / total_time, 6) if total_time else 0.0
summary_benchmark_row = {"notebook": "04_consolidacion_scoring_reporte_final", "source": "ALL", "step": "benchmark_general_summary", "duration_seconds": round(total_time, 6), "records_processed": records_processed, "records_per_second": signals_per_second, "status": "ok", "error_message": "", "notes": f"notebook_mas_lento={slowest_notebook}; fuente_mas_lenta={slowest_source}; senales_generadas={signals_generated}; senales_segundo={signals_per_second}", "benchmark_file": "04_benchmark_general.csv"}
benchmark_general = pd.concat([benchmark_general, pd.DataFrame([summary_benchmark_row])], ignore_index=True, sort=False)
benchmark_general_path = OUTPUT_DIR / "04_benchmark_general.csv"
benchmark_general.to_csv(benchmark_general_path, index=False, encoding="utf-8-sig")
print(f"Benchmark general guardado: {benchmark_general_path}")
print(f"Tiempo total estimado: {round(total_time, 3)} s; senales={signals_generated}; senales/seg={signals_per_second}")
display(benchmark_general.tail(20))


In [ ]:
def synthetic_names(base_names, size):
    names = []
    for i in range(size):
        base = base_names[i % len(base_names)]
        variant = i // len(base_names)
        if variant % 5 == 0:
            names.append(base)
        elif variant % 5 == 1:
            names.append(base.upper())
        elif variant % 5 == 2:
            names.append(f"{base} {variant}")
        elif variant % 5 == 3:
            names.append(" ".join(sorted(base.split())))
        else:
            names.append(normalize_name(base))
    return names
base_names = seeds_df["nombre_persona_input"].head(10).tolist() or ["Nombre Persona Prueba"]
synthetic_queries = synthetic_names(base_names, 10_000)
mock_records = synthetic_names(base_names + ["Registro Publico Mock", "Persona Sin Relacion"], 5_000)
mock_choices = [normalize_name(name) for name in mock_records]
start = time.perf_counter(); match_count = 0; best_scores = []
for query in tqdm(synthetic_queries, desc="Speed test matching 10k x 5k"):
    result = process.extractOne(normalize_name(query), mock_choices, scorer=fuzz.WRatio, score_cutoff=60)
    if result:
        match_count += 1; best_scores.append(float(result[1]))
elapsed = time.perf_counter() - start
speed_df = pd.DataFrame([{"test": "rapidfuzz_extractOne_10000_vs_5000", "synthetic_queries": len(synthetic_queries), "mock_records": len(mock_records), "nominal_comparisons": len(synthetic_queries) * len(mock_records), "duration_seconds": round(elapsed, 6), "queries_per_second": round(len(synthetic_queries) / elapsed, 2) if elapsed else 0, "matches_score_60_plus": match_count, "avg_best_score": round(sum(best_scores) / len(best_scores), 2) if best_scores else 0}])
speed_path = OUTPUT_DIR / "04_speed_test_matching.csv"
speed_df.to_csv(speed_path, index=False, encoding="utf-8-sig")
print(f"Speed test guardado: {speed_path}")
display(speed_df)
